# Multi-plant Disease Detector - training notebook (Kaggle)

Trains ONE MobileNetV2 model on **every plant-disease dataset you attach** (many crops, many diseases) plus an
**Unknown / not a leaf** class so the app can refuse images it doesn't understand.

**Before running**
1. Right panel -> *Add Input* -> attach as many plant-disease datasets as you like. Search for: `PlantVillage`,
   `New Plant Diseases Dataset`, `PlantDoc`, `rice leaf diseases`, `cassava leaf disease`, `wheat disease`,
   `apple / grape / cotton / sugarcane / maize leaf disease`, ... (the more crops, the wider the coverage).
2. *Settings -> Accelerator* -> **GPU T4 / P100**.
3. *Settings -> Internet* -> **On** (ImageNet weights + CIFAR-10 for the Unknown class).

No dataset is downloaded to your PC. Afterwards download `leaf_model.keras` + `class_names.json` from the **Output** tab.

In [ ]:
import os, json, random, re
from collections import defaultdict

# ---------------- CONFIG ----------------
INPUT_ROOT = "/kaggle/input"
CROPS = None              # None = ALL crops found. Or e.g. ["Tomato", "Potato", "Rice"]
MAX_PER_CLASS = 1200      # cap images per class (keeps training fast + balanced)
MIN_PER_CLASS = 40        # drop classes with fewer images than this
USE_UNKNOWN_CLASS = True  # adds 'Unknown___not_a_leaf' (uses CIFAR-10 photos)
IMG_SIZE = (224, 224)
BATCH = 32
EPOCHS_HEAD = 6
EPOCHS_FINE = 6
SEED = 42

## 1. Discover every class folder in every attached dataset

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png")
EXCLUDE_WORDS = ("grayscale", "segmented")           # PlantVillage duplicates in other colour modes
SKIP_PARENTS = {"train", "valid", "val", "validation", "test", "testing", "training", "images", "image",
                "color", "data", "dataset", "datasets", "raw", "plantvillage", "input"}

def norm(name):
    return re.sub(r"[\s\-]+", "_", name.strip()).strip("_")

def label_for(dirpath, root):
    name = norm(os.path.basename(dirpath))
    if "___" in name:                                  # already 'Crop___Disease'
        return name
    # otherwise use the nearest meaningful parent folder as the crop/dataset name
    parts = os.path.relpath(dirpath, root).split(os.sep)[:-1]
    for p in reversed(parts):
        if p.lower() not in SKIP_PARENTS:
            return f"{norm(p)}___{name}"
    return f"Plant___{name}"

def discover(root):
    by_label, display = defaultdict(list), {}
    for r, dirs, files in os.walk(root):
        if any(w in r.lower() for w in EXCLUDE_WORDS):
            continue
        imgs = [f for f in files if f.lower().endswith(IMG_EXT)]
        if len(imgs) >= 5:
            lab = label_for(r, root)
            key = lab.lower()
            display.setdefault(key, lab)
            by_label[key] += [os.path.join(r, f) for f in imgs]
    return by_label, display

by_label, display = discover(INPUT_ROOT)
print("Raw classes found:", len(by_label))

rng = random.Random(SEED)
class_names, class_files = [], {}
for key in sorted(by_label):
    lab = display[key]
    if CROPS and not any(lab.lower().startswith(c.lower()) for c in CROPS):
        continue
    files = sorted(set(by_label[key]))
    if len(files) < MIN_PER_CLASS:
        continue
    rng.shuffle(files)
    class_names.append(lab)
    class_files[lab] = files[:MAX_PER_CLASS]

assert class_names, "No classes found. Did you click 'Add Input' and attach a plant-disease dataset?"
crops = sorted({c.split("___")[0] for c in class_names})
print(f"{len(class_names)} disease classes across {len(crops)} plants")
print("Plants:", crops)
for c in class_names:
    print(f"  {c:60s} {len(class_files[c])}")

## 2. Stratified train/validation split

In [ ]:
paths, labels = [], []
for i, c in enumerate(class_names):
    for f in class_files[c]:
        paths.append(f); labels.append(i)

idx_by_class = defaultdict(list)
for i, l in enumerate(labels):
    idx_by_class[l].append(i)

train_idx, val_idx = [], []
for l, idxs in idx_by_class.items():
    rng.shuffle(idxs)
    cut = max(1, int(0.8 * len(idxs)))
    train_idx += idxs[:cut]; val_idx += idxs[cut:]
rng.shuffle(train_idx); rng.shuffle(val_idx)
print("plant images  train:", len(train_idx), " val:", len(val_idx))

## 3. tf.data pipelines (+ 'Unknown / not a leaf' class from CIFAR-10)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

print("TensorFlow:", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))
AUTOTUNE = tf.data.AUTOTUNE

def load(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(tf.round(img), tf.uint8)
    img.set_shape((*IMG_SIZE, 3))
    return img, label

def plant_ds(indices):
    p = [paths[i] for i in indices]; l = [labels[i] for i in indices]
    return tf.data.Dataset.from_tensor_slices((p, l)).map(load, num_parallel_calls=AUTOTUNE)

train_plants, val_plants = plant_ds(train_idx), plant_ds(val_idx)

unknown_ok = False
if USE_UNKNOWN_CLASS:
    try:
        (xtr, _), (xte, _) = tf.keras.datasets.cifar10.load_data()
        UNKNOWN = "Unknown___not_a_leaf"
        class_names.append(UNKNOWN)
        u = len(class_names) - 1
        def unk_ds(x):
            ds = tf.data.Dataset.from_tensor_slices(x)
            def f(im):
                im = tf.image.resize(im, IMG_SIZE)
                return tf.cast(tf.round(im), tf.uint8), tf.constant(u, tf.int64)
            return ds.map(f, num_parallel_calls=AUTOTUNE)
        n_unk = min(MAX_PER_CLASS * 2, len(xtr))
        train_unk, val_unk = unk_ds(xtr[:n_unk]), unk_ds(xte[:max(200, n_unk // 5)])
        train_plants = train_plants.concatenate(train_unk)
        val_plants = val_plants.concatenate(val_unk)
        unknown_ok = True
        print("Added Unknown class with", n_unk, "train images")
    except Exception as e:
        print("Could not add Unknown class (is Internet on?):", e)

def finish(ds, training):
    if training:
        ds = ds.shuffle(4096, seed=SEED)
    ds = ds.batch(BATCH).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds, val_ds = finish(train_plants, True), finish(val_plants, False)
NUM_CLASSES = len(class_names)
print("Total classes:", NUM_CLASSES)

# class weights to soften imbalance
counts = np.bincount(np.array([labels[i] for i in train_idx]), minlength=NUM_CLASSES).astype(float)
if unknown_ok: counts[-1] = n_unk
counts[counts == 0] = 1
cw = counts.sum() / (NUM_CLASSES * counts)
class_weight = {i: float(min(max(w, 0.3), 3.0)) for i, w in enumerate(cw)}

## 4. Model: MobileNetV2 (scaling + augmentation baked in)

In [ ]:
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
], name="augment")

base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = layers.Input((*IMG_SIZE, 3))                       # raw 0-255 pixels
x = augment(inputs)                                         # only active while training
x = layers.Rescaling(1/127.5, offset=-1, name="mobilenet_preprocess")(x)   # == MobileNetV2 preprocess_input
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
model = models.Model(inputs, outputs)

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 5. Train: head first, then fine-tune the top of MobileNetV2

In [ ]:
cb = [tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor="val_accuracy")]
h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=class_weight, callbacks=cb)

base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, class_weight=class_weight, callbacks=cb)

## 6. Evaluate

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for imgs, lab in val_ds:
    p = model.predict(imgs, verbose=0)
    y_true += lab.numpy().tolist(); y_pred += np.argmax(p, axis=1).tolist()

present = sorted(set(y_true) | set(y_pred))
print(classification_report(y_true, y_pred, labels=present,
                            target_names=[class_names[i] for i in present], digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=present)
side = max(8, len(present) * 0.28)
plt.figure(figsize=(side, side))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(len(present)), [class_names[i] for i in present], rotation=90, fontsize=5)
plt.yticks(range(len(present)), [class_names[i] for i in present], fontsize=5)
plt.title("Confusion matrix"); plt.colorbar(); plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150); plt.show()

acc = h1.history["accuracy"] + h2.history["accuracy"]
val = h1.history["val_accuracy"] + h2.history["val_accuracy"]
plt.plot(acc, label="train"); plt.plot(val, label="val"); plt.legend(); plt.title("Accuracy")
plt.savefig("/kaggle/working/accuracy_curve.png", dpi=150); plt.show()

# weakest classes (worth mentioning honestly in your report)
per = {}
for t, p in zip(y_true, y_pred):
    per.setdefault(t, []).append(t == p)
worst = sorted(((np.mean(v), class_names[k]) for k, v in per.items()))[:10]
print("Weakest classes:"); [print(f"  {a:.2f}  {b}") for a, b in worst]

## 7. Export (the only files you need to download)

In [ ]:
OUT = "/kaggle/working"
model.save(f"{OUT}/leaf_model.keras")
json.dump(class_names, open(f"{OUT}/class_names.json", "w"), indent=1)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
open(f"{OUT}/leaf_model.tflite", "wb").write(converter.convert())

for f in ["leaf_model.keras", "leaf_model.tflite", "class_names.json"]:
    print(f, round(os.path.getsize(f"{OUT}/{f}") / 1e6, 2), "MB")

## 8. Download
Output tab -> download `leaf_model.keras` and `class_names.json` (+ `.tflite`, `confusion_matrix.png`, `accuracy_curve.png`)
and put the first two in the project's `model/` folder.